# Day 5 · 실습 1. KLUE BERT로 한국어 뉴스 주제 분류하기

이번 실습에서는 **사전학습된 한국어 BERT를 실제 뉴스 데이터에 맞게 Fine-tuning**하고, 학습된 모델을 재사용하여 뉴스 제목의 주제를 분류합니다.

- **A형:** Hugging Face 모델·데이터 준비 → Tokenizer와 BERT 구조 확인 → 분류 Fine-tuning → 평가·오류 분석 → 모델 저장 및 예측
- **B형 1단계:** A형 코드를 참고해 새 뉴스 제목을 분류하는 함수 완성하기 (1문제)
- **B형 2단계:** 여러 뉴스 제목을 한 번에 분류하고 검토 대상으로 구분하기 (1문제)

**사용 모델:** [`klue/bert-base`](https://huggingface.co/klue/bert-base) · **공개 데이터:** [KLUE-YNAT](https://huggingface.co/datasets/klue/klue) (IT과학·경제·사회·생활문화·세계·스포츠·정치, 7개 클래스)

모델과 데이터는 코드에서 자동으로 다운로드합니다. **CUDA GPU(RTX 4090)를 사용하는 환경**을 전제로 하며, 처음 실행 시 인터넷 연결이 필요합니다. 이번 실습은 뉴스 제목의 **단일 정답 다중 클래스 분류**이며, B형에서는 A형에서 저장한 Fine-tuning 결과를 사용합니다.

## A형 · BERT 모델 구조 확인부터 Fine-tuning까지

### A-1. 실행 환경과 GPU 확인하기

사전학습 모델을 불러오고 학습하려면 `transformers`, `datasets` 등이 필요합니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **공개 모델과 데이터:** Hugging Face에서 별도 로그인 없이 다운로드합니다.
2. **GPU:** BERT의 학습에 CUDA GPU를 사용합니다.
3. **재현 가능성:** 난수 시드를 설정하여 데이터 샘플 선택이 일정하도록 합니다.

GPU 환경에 설치된 CUDA용 PyTorch는 그대로 사용하며, 아래 첫 셀에서는 관련 라이브러리만 설치합니다.

In [1]:
# 1. Hugging Face 모델·데이터를 다루는 라이브러리 설치
# GPU가 설치된 환경의 torch는 다시 설치하지 않습니다.
%pip -q install "transformers==4.57.1" "datasets==4.4.1" "scikit-learn>=1.4,<2" "pandas>=2.0,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 4090


In [3]:
# Hugging Face 다운로드 가속 기능 비활성화
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# 1. 필요한 라이브러리 불러오기
from pathlib import Path
import random

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# 2. GPU와 난수 시드 준비
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# BERT Fine-tuning은 GPU에서 진행합니다.
# CUDA가 없는 경우 조용히 CPU로 바꾸지 않고, 이 필수 조건을 바로 확인합니다.
assert torch.cuda.is_available(), "CUDA GPU가 필요합니다. GPU 환경에서 노트북을 실행하세요."
device = torch.device("cuda")

# Tensor 값을 확인할 때 지수 표기 대신 일반 숫자로 표시합니다.
torch.set_printoptions(precision=3, sci_mode=False)

print("사용 장치:", torch.cuda.get_device_name(0))
print("PyTorch 버전:", torch.__version__)

사용 장치: NVIDIA GeForce RTX 4090
PyTorch 버전: 2.8.0+cu128


### A-2. 한국어 뉴스 분류 데이터 준비하기

KLUE-YNAT에는 실제 한국어 뉴스 **제목(`title`)과 주제 정답(`label`)**이 들어 있습니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **데이터 구조:** 하나의 뉴스 제목에 0~6의 클래스 ID가 부여되어 있습니다.
2. **라벨 이름:** 클래스 ID를 실제 뉴스 주제에 대응시킵니다.
3. **학습 범위:** 원본 데이터와 공식 Validation을 구분한 채, 학습이 너무 길어지지 않도록 각 클래스에서 일부를 선택합니다.

데이터셋은 Hugging Face의 **공개 Parquet 파일**에서 자동 다운로드합니다. 학습용 샘플은 원본 Train에서, 평가용 샘플은 별도의 Validation에서 선택합니다.

In [4]:
# 1. 뉴스 주제 라벨 정의
# KLUE-YNAT 데이터셋에 정의된 0~6 라벨의 순서입니다.
LABEL_NAMES = ["IT과학", "경제", "사회", "생활문화", "세계", "스포츠", "정치"]
NUM_LABELS = len(LABEL_NAMES)

# 2. Hugging Face 공개 데이터 다운로드
# 데이터를 노트북에 직접 포함하지 않고, 실행 시 자동으로 내려받습니다.
DATA_BASE = "https://huggingface.co/datasets/klue/klue/resolve/main/ynat"
data_files = {
    "train": f"{DATA_BASE}/train-00000-of-00001.parquet",
    "validation": f"{DATA_BASE}/validation-00000-of-00001.parquet",
}
raw = load_dataset("parquet", data_files=data_files)

# 필요한 열만 선택합니다. title: 뉴스 제목 / label: 주제 클래스 ID
data_train_all = raw["train"].to_pandas()[["title", "label"]]
data_valid_all = raw["validation"].to_pandas()[["title", "label"]]

# 3. 수업 시간에 맞게 클래스별로 같은 개수의 데이터를 추출합니다.
# 데이터를 샘플링하는 방법은 이번 실습의 구현 문제가 아니므로 완성 코드로 제공합니다.
TRAIN_PER_CLASS = 400   # 7개 클래스 × 400건 = Train 2,800건
VALID_PER_CLASS = 80    # 7개 클래스 ×  80건 = Validation 560건

train_df = (
    data_train_all.groupby("label", group_keys=False)
    .sample(n=TRAIN_PER_CLASS, random_state=SEED)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
valid_df = (
    data_valid_all.groupby("label", group_keys=False)
    .sample(n=VALID_PER_CLASS, random_state=SEED)
    .reset_index(drop=True)
)

# 4. 데이터 확인
print("원본 Train/Validation:", len(data_train_all), "/", len(data_valid_all))
print("이번 Train/Validation:", len(train_df), "/", len(valid_df))
print("\n클래스별 Train 데이터 수")
print(train_df["label"].map(lambda x: LABEL_NAMES[x]).value_counts().sort_index())

# 뉴스 제목·정답 라벨을 나란히 확인합니다.
display(train_df.assign(카테고리=train_df["label"].map(lambda x: LABEL_NAMES[x]))[["title", "카테고리"]].head(7))

원본 Train/Validation: 45678 / 9107
이번 Train/Validation: 2800 / 560

클래스별 Train 데이터 수
label
IT과학    400
경제      400
사회      400
생활문화    400
세계      400
스포츠     400
정치      400
Name: count, dtype: int64


,title,카테고리
0,삼성카드 핸드메이드 유통 플랫폼 백패커와 업무제휴,사회
1,현대차증권 삼성엔지니어링 2분기 호실적…목표주가↑,경제
2,종아리 통증 최지만 2경기 연속 결장…탬파베이 4연패 탈출,스포츠
3,현대상선 용선료 협상 타결돼도 산 넘어 산,사회
4,일대일로 외연 넓히는 중국…스위스도 일대일로 협력종합,세계
5,눈보라 휘말린 러 헬기 동시베리아서 불시착…3명 부상,세계
6,서울 사흘 연속 미세먼지 주의보 발령…외출 자제,생활문화


### A-3. Tokenizer로 BERT 입력 만들기

BERT는 문자열을 바로 계산하지 않고 Tokenizer가 만든 정수 ID를 입력받습니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **`input_ids`:** `[CLS]`, `[SEP]` 등의 특수 토큰을 포함한 토큰 ID입니다.
2. **`attention_mask`:** 실제 입력 위치는 1, Padding 위치는 0입니다.
3. **`token_type_ids`:** 하나의 문장만 입력할 때는 일반적으로 모두 0입니다.

`padding="max_length"`는 모든 입력의 길이를 지정한 길이로 맞추며, `truncation=True`는 길이를 초과하는 부분을 자릅니다.

In [5]:
# 1. 모델과 Tokenizer 자동 다운로드
MODEL_ID = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2. 실제 뉴스 제목 하나를 BERT 입력으로 변환
sample_title = train_df.iloc[0]["title"]

########## 핵심 1: 문장을 input_ids·attention_mask·token_type_ids로 변환합니다.
# max_length는 [CLS], [SEP] 등을 포함한 전체 토큰 길이입니다.
sample_inputs = tokenizer(
    sample_title,
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt",
)

# 3. 토큰의 실제 문자열과 입력 Tensor 확인
sample_tokens = tokenizer.convert_ids_to_tokens(sample_inputs["input_ids"][0].tolist())
print("원본 뉴스 제목:", sample_title)
print("토큰 문자열:", sample_tokens)
print("input_ids Shape:", tuple(sample_inputs["input_ids"].shape))
print("attention_mask:", sample_inputs["attention_mask"][0].tolist())
print("token_type_ids:", sample_inputs["token_type_ids"][0].tolist())

원본 뉴스 제목: 삼성카드 핸드메이드 유통 플랫폼 백패커와 업무제휴
토큰 문자열: ['[CLS]', '삼성카드', '핸드', '##메이드', '유통', '플랫폼', '백', '##패', '##커', '##와', '업무', '##제', '##휴', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids Shape: (1, 32)
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### A-4. BERT Encoder와 Classification Head의 출력 확인하기

사전학습된 BERT Encoder는 **토큰별 문맥 표현**을 생성하고, 새 Classification Head는 **7개 클래스의 Logits**를 계산합니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **BERT Base:** Encoder Layer 12개, Hidden Size 768, Attention Head 12개입니다.
2. **문장 표현:** `[CLS]` 위치의 출력과 Pooler 출력을 확인합니다.
3. **분류 결과:** Classification Head의 출력 Shape는 `(문장 수, 7)`입니다.

**중요:** 처음 연결된 Classification Head는 아직 학습하지 않은 가중치로 초기화됩니다. 새로 초기화된 가중치에 관한 안내 메시지는 정상이며, **Fine-tuning 전의 점수를 실제 분류 성능으로 해석하면 안 됩니다.**

In [6]:
# 1. 사전학습된 BERT + 새로운 7개 클래스 Classification Head 준비
# id2label, label2id는 모델 출력 번호를 실제 카테고리 이름과 연결합니다.
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    use_safetensors=True,
).to(device)

print("Encoder Layer 수:", model.config.num_hidden_layers)
print("Hidden Size:", model.config.hidden_size)
print("Attention Head 수:", model.config.num_attention_heads)
print("Classification Head 가중치 Shape:", tuple(model.classifier.weight.shape))

# 2. 학습 전에 입력·출력 구조를 확인합니다.
model.eval()
example_inputs = sample_inputs.to(device)

with torch.inference_mode():
    ########## 핵심 2: BERT Encoder의 토큰별 출력과 [CLS] 표현을 확인합니다.
    bert_outputs = model.bert(**example_inputs)
    last_hidden = bert_outputs.last_hidden_state
    cls_vector = last_hidden[:, 0, :]
    pooled_vector = bert_outputs.pooler_output

    ########## 핵심 3: Classification Head에서 뉴스 주제별 Logits를 계산합니다.
    classification_logits = model(**example_inputs).logits

print("Encoder 출력:", tuple(last_hidden.shape))
print("[CLS] 위치의 벡터:", tuple(cls_vector.shape))
print("Pooler 출력:", tuple(pooled_vector.shape))
print("분류 Logits:", tuple(classification_logits.shape))
print("분류 Head의 초기 Logits:", classification_logits[0].cpu())

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoder Layer 수: 12
Hidden Size: 768
Attention Head 수: 12
Classification Head 가중치 Shape: (7, 768)
Encoder 출력: (1, 32, 768)
[CLS] 위치의 벡터: (1, 768)
Pooler 출력: (1, 768)
분류 Logits: (1, 7)
분류 Head의 초기 Logits: tensor([ 0.043,  0.021,  0.128, -0.266, -0.096,  0.182,  0.224])


### A-5. 학습에 사용할 Batch 만들기

여러 문장을 Batch로 묶으면 GPU에서 효율적으로 처리할 수 있습니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **Max Length:** 뉴스 제목을 최대 64토큰 길이로 변환합니다.
2. **Train / Validation:** 두 데이터는 서로 다른 원본 분할에서 가져왔습니다.
3. **Batch:** `DataLoader`가 각 문장과 정답 라벨을 묶어 전달합니다.

이 셀의 데이터 로딩 준비 코드는 강사가 모두 제공한 부분이므로, 각 Tensor의 Shape와 역할만 확인하면 됩니다.

In [7]:
# 1. Fine-tuning에 사용할 주요 설정
MAX_LENGTH = 64
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5

# 2. 텍스트와 라벨을 TensorDataset으로 준비하는 공통 함수
def make_loader(frame, shuffle=False):
    # 여러 뉴스 제목을 한 번에 토큰화하고, 같은 길이의 Tensor로 변환합니다.
    inputs = tokenizer(
        frame["title"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    # 정답 클래스 ID는 Cross-Entropy Loss가 사용하는 정수형 Tensor입니다.
    labels = torch.tensor(frame["label"].tolist(), dtype=torch.long)

    data = TensorDataset(
        inputs["input_ids"],
        inputs["attention_mask"],
        inputs["token_type_ids"],
        labels,
    )
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(train_df, shuffle=True)
valid_loader = make_loader(valid_df, shuffle=False)

# 3. Batch 하나의 Shape 확인
batch_input_ids, batch_masks, batch_types, batch_labels = next(iter(train_loader))
print("input_ids:", tuple(batch_input_ids.shape))
print("attention_mask:", tuple(batch_masks.shape))
print("token_type_ids:", tuple(batch_types.shape))
print("labels:", tuple(batch_labels.shape))
print("학습 Batch 수:", len(train_loader))

input_ids: (16, 64)
attention_mask: (16, 64)
token_type_ids: (16, 64)
labels: (16,)
학습 Batch 수: 175


### A-6. BERT Full Fine-tuning 진행하기

이번 단계에서는 **사전학습된 BERT와 새 Classification Head의 가중치를 함께 갱신**합니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **Forward:** 입력을 모델에 전달하면 클래스별 `logits`와 Cross-Entropy `loss`가 계산됩니다.
2. **Backward:** `loss.backward()`로 기울기를 계산합니다.
3. **Optimizer:** `optimizer.step()`으로 BERT와 분류 계층의 가중치를 갱신합니다.

Hugging Face의 `AutoModelForSequenceClassification`은 `labels`를 입력받으면 분류 손실을 계산해 `outputs.loss`로 제공합니다. 이때 손실 계산에 Softmax 확률이 아닌 **Logits를 직접 사용**한다는 점에 주목하세요.

In [8]:
# 1. Optimizer 설정
# 모델 전체 파라미터를 전달하므로 BERT Encoder와 Classification Head가 함께 학습됩니다.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# 2. 지정한 Epoch만큼 학습합니다.
for epoch in range(EPOCHS):
    model.train()  # Dropout 등을 학습 모드로 설정합니다.
    total_loss = 0.0

    for input_ids, attention_mask, token_type_ids, labels in tqdm(
        train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=False
    ):
        # 이번 Batch를 GPU로 이동합니다.
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        token_type_ids = token_type_ids.to(device)
        labels = labels.to(device)

        # 직전 Batch에서 계산한 기울기를 초기화합니다.
        optimizer.zero_grad()

        ########## 핵심 1: BERT와 Classification Head를 통과해 Logits·Loss를 계산합니다.
        # labels를 함께 전달하면 모델 내부에서 Cross-Entropy Loss를 계산합니다.
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )
        loss = outputs.loss

        ########## 핵심 2: 손실을 기준으로 모델 가중치들의 기울기를 계산합니다.
        loss.backward()

        ########## 핵심 3: 사전학습된 BERT와 새 분류 계층의 가중치를 갱신합니다.
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1} · 평균 Train Loss: {total_loss / len(train_loader):.4f}")

print("Fine-tuning 완료")

Epoch 1/2:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 1 · 평균 Train Loss: 0.8505


Epoch 2/2:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 2 · 평균 Train Loss: 0.3794
Fine-tuning 완료


### A-7. 분류 성능과 오분류 문장 확인하기

학습된 모델을 Validation 데이터에 적용하고 **Accuracy, Macro F1, Confusion Matrix**를 확인합니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **`model.eval()` + `torch.inference_mode()`:** 평가에서는 가중치를 갱신하지 않습니다.
2. **클래스 예측:** Logits에서 가장 높은 점수의 클래스 ID를 구합니다.
3. **오류 분석:** 실제 정답과 예측 클래스가 다른 뉴스 제목을 확인합니다.

뉴스 제목에는 서로 다른 분야의 단어가 함께 등장할 수 있으므로, 오분류된 사례를 실제 문장과 함께 살펴보는 것이 중요합니다.

In [9]:
# 1. Validation에 대한 예측 결과 모으기
model.eval()
all_predictions = []
all_targets = []
all_confidences = []

with torch.inference_mode():
    for input_ids, attention_mask, token_type_ids, labels in valid_loader:
        outputs = model(
            input_ids=input_ids.to(device),
            attention_mask=attention_mask.to(device),
            token_type_ids=token_type_ids.to(device),
        )

        ########## 핵심: Logits → Softmax → 예측 클래스 ID
        probabilities = torch.softmax(outputs.logits, dim=-1)
        confidence, predicted_ids = probabilities.max(dim=-1)

        all_predictions.extend(predicted_ids.cpu().tolist())
        all_confidences.extend(confidence.cpu().tolist())
        all_targets.extend(labels.tolist())

# 2. 전체 정확도와 클래스별 F1을 동일 비중으로 평균한 Macro F1 확인
acc = accuracy_score(all_targets, all_predictions)
macro_f1 = f1_score(all_targets, all_predictions, average="macro", labels=list(range(NUM_LABELS)))
print(f"Validation Accuracy: {acc:.3f}")
print(f"Validation Macro F1: {macro_f1:.3f}")

# 3. 실제 클래스(행)와 예측 클래스(열)의 혼동행렬 확인
cm = confusion_matrix(all_targets, all_predictions, labels=list(range(NUM_LABELS)))
cm_df = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)
print("\nConfusion Matrix (행: 실제 / 열: 예측)")
display(cm_df)

# 4. 실제 문장을 함께 확인하여 오분류된 사례 분석
results_df = valid_df.copy()
results_df["실제 카테고리"] = [LABEL_NAMES[i] for i in all_targets]
results_df["예측 카테고리"] = [LABEL_NAMES[i] for i in all_predictions]
results_df["예측 점수"] = [round(x, 3) for x in all_confidences]

errors_df = results_df[results_df["label"] != all_predictions]
print("오분류 건수:", len(errors_df))
display(errors_df[["title", "실제 카테고리", "예측 카테고리", "예측 점수"]].head(10))

Validation Accuracy: 0.845
Validation Macro F1: 0.849

Confusion Matrix (행: 실제 / 열: 예측)


,IT과학,경제,사회,생활문화,세계,스포츠,정치
IT과학,62,10,5,3,0,0,0
경제,2,65,10,1,1,1,0
사회,2,7,68,1,1,0,1
생활문화,0,1,10,69,0,0,0
세계,1,3,2,3,67,3,1
스포츠,0,0,1,0,0,77,2
정치,0,0,12,0,3,0,65


오분류 건수: 87


,title,실제 카테고리,예측 카테고리,예측 점수
1,삼성 IM부분 영업익 3조대 선방…갤S8·갤노트8 효과,IT과학,경제,0.978
6,달로 향한 최초 비글 스누피와 함께한 현대미술 여행,IT과학,생활문화,0.972
7,티엘아이 eMMC 사업부 철수…IoT 사업부 물적분할,IT과학,경제,0.849
9,인크루트 로봇직원 알리사 채용…사번은 24365,IT과학,경제,0.367
10,울산교육청 울산대 등 4개 대학과 AI 융합교육 전문교사 양성,IT과학,사회,0.902
15,글로벌 TV시장 16% 축소…삼성·LG는 판매 확대,IT과학,경제,0.867
16,디지털미디어 소통역량 강화방안 발표하는 방통위,IT과학,사회,0.418
21,詩큐레이션 앱 시요일 이용자 10만명 돌파,IT과학,생활문화,0.856
26,이마트몰 카카오톡 장보기 서비스 개시,IT과학,경제,0.615
29,한국MS 포르자 호라이즌4 판매,IT과학,경제,0.309


### A-8. Fine-tuning된 모델 저장하고 뉴스 한 건 분류하기

학습한 가중치와 Tokenizer를 저장한 뒤, 새로운 뉴스 제목을 입력해 분류 결과를 확인합니다. **아래 코드에서 다음 세 가지를 확인하세요.**

1. **모델 저장:** 사전학습 모델이 아니라 **Fine-tuning 완료 모델**의 가중치를 저장합니다.
2. **예측 단계:** `tokenizer → model → logits → softmax → class_id` 순서로 계산합니다.
3. **결과 해석:** 클래스 ID를 실제 주제 이름으로 바꾸고 해당 클래스의 예측 점수를 확인합니다.

이 셀의 핵심 연산은 B형에서 그대로 재사용합니다.

In [10]:
# 1. Fine-tuning된 모델과 Tokenizer 저장
SAVE_DIR = Path("outputs/klue_bert_ynat_finetuned")
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print("모델 저장 경로:", SAVE_DIR.resolve())

# 2. 처음 보는 뉴스 제목 한 건 분류
demo_title = "국내 반도체 신기술 연구 성과 발표"
model.eval()

########## 핵심 1: 입력 문자열을 BERT 입력 Tensor로 변환해 GPU로 보냅니다.
demo_inputs = tokenizer(
    demo_title,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    ########## 핵심 2: 분류 계층에서 7개 클래스의 Logits를 계산합니다.
    demo_logits = model(**demo_inputs).logits

    ########## 핵심 3: Logits를 확률로 변환합니다.
    demo_probs = torch.softmax(demo_logits, dim=-1)

########## 핵심 4: 가장 높은 점수의 클래스 ID와 예측 점수를 가져옵니다.
demo_class_id = int(demo_probs.argmax(dim=-1).item())
demo_score = float(demo_probs[0, demo_class_id].item())

print("뉴스 제목:", demo_title)
print("예측 카테고리:", LABEL_NAMES[demo_class_id])
print(f"모델 예측 점수: {demo_score:.3f}")
# Softmax 예측 점수는 '실제로 정답일 확률'이 보정되어 있다는 뜻은 아닙니다.

모델 저장 경로: /workspace/outputs/klue_bert_ynat_finetuned
뉴스 제목: 국내 반도체 신기술 연구 성과 발표
예측 카테고리: IT과학
모델 예측 점수: 0.867


## B형 1단계 · 새 뉴스 제목을 분류하는 함수 완성하기

A-8에서 배운 **Tokenizer → Logits → Softmax → 클래스 ID 선택** 코드를 재사용하여 `predict_one_title()` 함수를 완성하세요.

- **구현 범위:** A-8의 핵심 연산을 함수 안에 옮기기 (코드 재현 문제)
- **참고 셀:** A-8의 `demo_inputs`, `demo_logits`, `demo_probs`, `demo_class_id`, `demo_score`
- **입력:** 뉴스 제목 문자열 `title`
- **출력:** `(예측 카테고리 이름, 예측 점수)`

B형에서는 모델을 다시 학습하지 않습니다. A형에서 저장한 Fine-tuning 완료 모델을 불러옵니다.

In [11]:
# 1. A형에서 저장한 Fine-tuning 완료 모델 불러오기

model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIR, use_safetensors=True
).to(device)
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model.eval()

# 2. 새로운 제목을 분류하는 함수 완성하기
def predict_one_title(title):
    
    encoded = tokenizer(
        demo_title,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
    
        logits = model(**demo_inputs).logits
        probs = torch.softmax(logits, dim=-1)

    class_id = int(probs.argmax(dim=-1).item())
    score = float(probs[0, class_id].item())

    return LABEL_NAMES[class_id], score

In [12]:
# 3. B형 1단계 결과 확인
# A-8에서 사용한 제목을 넣으면 같은 예측 결과가 나와야 합니다.
b1_label, b1_score = predict_one_title(demo_title)
assert b1_label == LABEL_NAMES[demo_class_id]
assert abs(b1_score - demo_score) < 1e-5
print("B형 1단계 통과: A형의 분류 결과를 정확하게 재현했습니다.")

# 새로운 뉴스 제목에도 함수를 사용해 보세요.
new_title = "국내 프로축구 대표팀 평가전 최종 명단 발표"
print("새로운 뉴스 예측:", predict_one_title(new_title))

B형 1단계 통과: A형의 분류 결과를 정확하게 재현했습니다.
새로운 뉴스 예측: ('IT과학', 0.8667155504226685)


## B형 2단계 · 여러 뉴스 제목을 한 번에 분류하기

A-8과 B형 1단계에서 배운 분류 코드를 **여러 문장에 적용하는 함수**로 조합합니다.

- **구현 범위:** `model(**encoded).logits`와 `torch.softmax()`의 핵심 연산 2줄
- **입력 준비:** `titles` 목록을 한 번에 Tokenizer에 전달하는 코드는 제공되어 있습니다.
- **결과 정리:** 여러 문장의 예측값을 표로 만들고, 점수가 낮은 항목을 표시하는 코드는 제공되어 있습니다.

`padding=True`는 Batch 안에서 가장 긴 문장에 맞춰 길이를 조정하는 설정입니다. 이번 과제에서 새롭게 사용하는 코드이므로 학생이 직접 작성할 필요는 없습니다.

**검토 기준:** 예측 점수가 0.55보다 낮으면 `검토 필요`로 표시합니다. 이 값은 교육용으로 정한 임의의 기준이며, 실제 정답률을 보장하는 확률은 아닙니다.

In [15]:
# 1. A형 코드가 처리하던 문장 1개를 여러 뉴스 제목으로 확장
news_titles = [
    "국내 반도체 신기술 연구 성과 발표",
    "기준금리 변동에 금융시장 동향 주목",
    "국가대표 축구팀 국제대회 준비 본격화",
    "해외 정상회담에서 외교 협력 논의",
    "지역 주민 대상 문화 축제 개최",
    "새로운 복지 정책 시행에 대한 시민 반응",
]


def predict_many_titles(titles, review_threshold=0.55):
    # 여러 뉴스 제목을 한 번에 Tokenizer로 처리합니다.
    encoded = tokenizer(
        titles,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)

    # 아래는 아직 배우지 않은 여러 결과를 표로 정리하는 코드이므로 완성해 두었습니다.
    scores, class_ids = probs.max(dim=-1)  # 문장별 가장 높은 예측 점수와 클래스 ID
    rows = []
    for title, class_id, score in zip(titles, class_ids.tolist(), scores.tolist()):
        rows.append({
            "뉴스 제목": title,
            "예측 카테고리": LABEL_NAMES[class_id],
            "예측 점수": round(score, 3),
            "검토 여부": "검토 필요" if score < review_threshold else "자동 분류",
        })
    return pd.DataFrame(rows)

In [16]:
# 2. B형 2단계 결과 확인
batch_results = predict_many_titles(news_titles)
assert len(batch_results) == len(news_titles)
assert set(batch_results["예측 카테고리"]).issubset(set(LABEL_NAMES))
assert batch_results["예측 점수"].between(0.0, 1.0).all()
print("B형 2단계 통과: 여러 뉴스 제목을 자동 분류했습니다.")
display(batch_results)

B형 2단계 통과: 여러 뉴스 제목을 자동 분류했습니다.


,뉴스 제목,예측 카테고리,예측 점수,검토 여부
0,국내 반도체 신기술 연구 성과 발표,IT과학,0.867,자동 분류
1,기준금리 변동에 금융시장 동향 주목,경제,0.903,자동 분류
2,국가대표 축구팀 국제대회 준비 본격화,스포츠,0.977,자동 분류
3,해외 정상회담에서 외교 협력 논의,정치,0.923,자동 분류
4,지역 주민 대상 문화 축제 개최,생활문화,0.768,자동 분류
5,새로운 복지 정책 시행에 대한 시민 반응,사회,0.883,자동 분류


### 실습 마무리

이번 실습에서는 **한국어 사전학습 BERT를 실제 분류 작업에 Fine-tuning하고, 저장된 모델을 재사용하는 과정**을 확인했습니다.

1. **Tokenizer:** 뉴스 제목을 `input_ids`·`attention_mask` 등의 BERT 입력으로 변환했습니다.
2. **BERT + Classification Head:** Encoder의 문맥 표현으로 7개 클래스의 Logits를 계산했습니다.
3. **Full Fine-tuning:** 분류 정답 라벨을 이용해 BERT와 Classification Head의 가중치를 함께 갱신했습니다.
4. **평가:** Accuracy·Macro F1·Confusion Matrix와 오분류 사례를 확인했습니다.
5. **응용:** 저장된 모델을 불러와 단일 뉴스와 여러 뉴스를 분류했습니다.

**핵심:** 사전학습된 BERT는 문맥 표현을 만드는 출발점이며, **Fine-tuning으로 분류 작업에 맞게 가중치를 조정한 후 실제 텍스트 데이터에 적용**할 수 있습니다.